# 02 · Feature Engineering (pandas)

Turn the clean data into a **model-ready** table, without throwing anything away.

- **Input:**  `data/products_clean.csv`  (all ~70k rows)
- **Output:** `data/products_features.csv`  (all rows + new feature columns + an `is_trainable` flag)

Design decision: we **keep all 70k rows** in one file. A flag marks the rows the
model will use; the dashboards can read the same file. The model (notebook 04)
filters to `is_trainable`; nothing is lost here.

## 1. Setup & load

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 50)
DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
df = pd.read_csv(DATA_DIR / "products_clean.csv", dtype={"code": "string"})
print("Loaded:", df.shape)

## 2. Encode the target & flag the trainable rows

XGBoost needs a **numeric** target, so map the grade A–E → 0–4. Rows with no grade
stay null (they simply aren't trainable). The model needs both a known grade
**and** the core nutrients, so `is_trainable` combines those two conditions.

In [ ]:
GRADE_MAP = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
df["grade_encoded"] = df["nutriscore_grade"].map(GRADE_MAP).astype("Int64")

# has_full_nutrition came from notebook 01; trainable = full nutrition AND a known grade
df["is_trainable"] = df["has_full_nutrition"] & df["nutriscore_grade"].notna()

print("Trainable rows:", int(df["is_trainable"].sum()), "of", len(df))
print("\nGrade distribution (trainable only):")
print(df.loc[df["is_trainable"], "nutriscore_grade"].value_counts().sort_index().to_string())

## 3. Derived features

Small features that capture *composition*, not just raw amounts. Helpers keep a
missing input as missing (`NaN`) instead of silently turning it into a 0/1, so
XGBoost still sees "unknown" where it's genuinely unknown.

In [ ]:
# --- ratio features: how much of the energy is sugar / protein ---
energy = df["energy_kcal_100g"].replace(0, np.nan)          # avoid divide-by-zero
df["sugar_to_energy"]   = (df["sugars_100g"]   / energy).round(4)
df["protein_to_energy"] = (df["proteins_100g"] / energy).round(4)

# --- "good minus bad" nutrient balance (missing treated as 0 for this score) ---
df["good_minus_bad"] = (
    df["proteins_100g"].fillna(0) + df["fiber_100g"].fillna(0)
    - df["sugars_100g"].fillna(0) - df["saturated_fat_100g"].fillna(0)
).round(2)

# --- binary "high" flags (UK FSA traffic-light thresholds); NaN stays NaN ---
def high_flag(series, threshold):
    return (series > threshold).astype("Int64").where(series.notna())

df["is_high_sugar"]      = high_flag(df["sugars_100g"], 22.5)
df["is_high_salt"]       = high_flag(df["salt_100g"], 1.5)
df["is_high_sat_fat"]    = high_flag(df["saturated_fat_100g"], 5.0)
df["is_ultra_processed"] = (df["nova_group"] == 4).astype("Int64").where(df["nova_group"].notna())

df[["code", "sugar_to_energy", "good_minus_bad",
    "is_high_sugar", "is_ultra_processed"]].head()

## 4. Define the feature list

The columns the model will learn from. Note two deliberate exclusions:
- **`nutriscore_score`** — the grade is *computed* from it, so using it is target
  leakage (the model would cheat). Excluded.
- **`sodium_100g`** — it's just `salt / 2.5`, so it duplicates `salt_100g`. We keep
  salt and drop sodium to avoid a redundant feature.

In [ ]:
FEATURES = [
    # raw nutrients
    "energy_kcal_100g", "sugars_100g", "fat_100g", "saturated_fat_100g",
    "salt_100g", "proteins_100g", "fiber_100g",
    # processing / additives
    "nova_group", "additives_n",
    # derived
    "sugar_to_energy", "protein_to_energy", "good_minus_bad",
    "is_high_sugar", "is_high_salt", "is_high_sat_fat", "is_ultra_processed",
]
TARGET = "grade_encoded"

print(f"{len(FEATURES)} features:")
for f in FEATURES:
    print("  -", f)
print("\ntarget:", TARGET, "(A–E encoded 0–4)")

## 5. Save the model-ready file (all rows kept)

In [ ]:
out = DATA_DIR / "products_features.csv"
df.to_csv(out, index=False)
print(f"Saved {len(df):,} rows -> {out}")
print("New columns added:",
      ["grade_encoded", "is_trainable", "sugar_to_energy", "protein_to_energy",
       "good_minus_bad", "is_high_sugar", "is_high_salt", "is_high_sat_fat", "is_ultra_processed"])

## 6. Quick stratified-split preview

A sanity check that a stratified split keeps the grade balance in both train and
test. The real split + training happens in notebook 04 — this is just a preview.

In [ ]:
from sklearn.model_selection import train_test_split

model_df = df[df["is_trainable"]].copy()
X = model_df[FEATURES]
y = model_df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {len(X_train):,}   Test: {len(X_test):,}")
print("\nClass balance preserved (proportions):")
prev = pd.DataFrame({
    "train_%": (y_train.value_counts(normalize=True).sort_index()*100).round(1),
    "test_%":  (y_test.value_counts(normalize=True).sort_index()*100).round(1),
})
prev.index = ["A", "B", "C", "D", "E"]
print(prev.to_string())

## What we did

1. Encoded the grade A–E → 0–4 and flagged the trainable rows (`is_trainable`)
2. Built ratio, balance, and high/low flag features (missing stays missing)
3. Defined the feature list and excluded `nutriscore_score` (leakage) and
   `sodium_100g` (redundant with salt)
4. Saved `products_features.csv` — **all rows kept**, model filters via the flag
5. Previewed a stratified split to confirm class balance holds

**Next:** `04_modelling.ipynb` reads this file, filters to `is_trainable`, trains
XGBoost on `FEATURES → grade_encoded`, and evaluates it.